# LMA Phase 1 — Complete Pipeline (Scrape + Clean + Tokenize)

Runs on **Colab CPU**. All data on Google Drive.

**Manual data sources (for 20% requirement):**
- 30 Hindi RSS news feeds (BBC, NDTV, Aaj Tak, Dainik Jagran, Amar Ujala, etc.)
- 15 Nepali RSS news feeds (Setopati, Onlinekhabar, Ratopati, Kantipur, etc.)
- Hindi Wikisource (API)
- Nepali Wikisource (HTML fallback)
- Curated Wikipedia articles (131 Hindi, 63 Nepali topics)
- Kavitakosh (Hindi poetry/literature)
- Existing OCR + webcrawl data

**Pipeline:**
0. Setup (mount Drive, unzip code, deps)
1. Check existing data
2. Clean Hindi downloaded (C4 — already on Drive)
3. Scrape ALL manual sources (RSS + Wikisource + Wikipedia + Kavitakosh)
4. Clean all manual data
5. Check sizes + cap downloaded to 20% rule
6. Merge + splits
7. Train BPE tokenizers
8. Encode + stats
9. Examples

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/LMA_Project'
import os; os.makedirs(PROJ, exist_ok=True)
print('Drive mounted at', PROJ)

In [ ]:
import zipfile, os, sys
zip_path = f'{PROJ}/lma_code.zip'
dest = '/content/lma_code'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    print(f'Unzipped to {dest}')
else:
    print(f'ERROR: {zip_path} not found!')

In [ ]:
sys.path.insert(0, '/content/lma_code')
import common.clean, common.dedup, common.splits, common.stats, common.collect, common.langfilter
from hindi.tokenizer.tokenizer import Tokenizer as HiTok
from nepali.tokenizer.tokenizer import Tokenizer as NeTok
print('All imports OK')

In [ ]:
!pip install -q regex datasets beautifulsoup4 lxml
print('deps installed')

## 1. Check existing data on Drive

In [ ]:
import os, glob
PROJ = '/content/drive/MyDrive/LMA_Project'
for lang in ['hindi', 'nepali']:
    print(f'\n--- {lang.upper()} ---')
    for subdir in ['raw/downloaded', 'raw/manual', 'processed/clean', 'processed/clean_manual']:
        d = f'{PROJ}/{lang}/{subdir}'
        if not os.path.isdir(d): continue
        total = 0; files = 0
        for root, _, fs in os.walk(d):
            for f in fs:
                if f.endswith('.jsonl'):
                    total += os.path.getsize(os.path.join(root, f)); files += 1
        if files > 0:
            print(f'  {subdir}: {files} files, {total/1e6:.1f} MB')

## 2. Clean Hindi downloaded (C4 + wiki — already on Drive)
Nepali downloaded is already cleaned (10.78 GB) — skip.

In [ ]:
import glob, subprocess, sys, os
existing = glob.glob('/content/drive/MyDrive/LMA_Project/hindi/processed/clean/hindi_clean_*.jsonl')
if len(existing) > 2:
    print(f'Hindi downloaded already cleaned ({len(existing)} shards) — skipping')
else:
    out = '/content/drive/MyDrive/LMA_Project/hindi/processed/clean'
    for f in sorted(glob.glob('/content/drive/MyDrive/LMA_Project/hindi/raw/downloaded/*.jsonl')):
        print('cleaning', os.path.basename(f))
        subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'hindi',
                        '--input', f, '--output-dir', out,
                        '--source-tag', 'downloaded', '--shard-size', '250000',
                        '--min-chars', '80'], cwd='/content/lma_code')
    print('Hindi downloaded cleaning done')

## 3. Scrape ALL manual sources

**Sources:**
- **RSS news feeds:** 30 Hindi + 15 Nepali feeds (articles scraped from each)
- **Wikisource:** Hindi (API) + Nepali (HTML fallback)
- **Wikipedia curated:** 131 Hindi + 63 Nepali topic articles
- **Kavitakosh:** Hindi poetry/literature site

All tagged `source: manual`. All **original-language** texts, NOT translations.

Run Hindi and Nepali in **separate sessions** for parallelism.

In [ ]:
# Scrape HINDI manual data from ALL sources
!cd /content/lma_code && python scripts/scrape_manual.py --lang hindi \
    --output-dir /content/drive/MyDrive/LMA_Project/hindi/raw/manual \
    --sources rss,wikisource,wikipedia,kavitakosh --max-articles 300

In [ ]:
# Scrape NEPALI manual data from ALL sources
!cd /content/lma_code && python scripts/scrape_manual.py --lang nepali \
    --output-dir /content/drive/MyDrive/LMA_Project/nepali/raw/manual \
    --sources rss,wikisource,wikipedia --max-articles 300

In [ ]:
# Verify scraped text is genuine Devanagari (not English/transliteration)
import json, glob, os
for lang in ['hindi', 'nepali']:
    print(f'\n===== {lang.upper()} SAMPLES =====')
    base = f'/content/drive/MyDrive/LMA_Project/{lang}/raw/manual'
    for root, _, files in os.walk(base):
        for fn in sorted(files):
            if not fn.endswith('.jsonl'): continue
            fp = os.path.join(root, fn)
            with open(fp) as f:
                line = f.readline()
                if line:
                    rec = json.loads(line)
                    rel = fp.replace(f'/content/drive/MyDrive/LMA_Project/{lang}/raw/manual/', '')
                    print(f'  [{rel}] {rec.get("title", rec.get("url", "?"))[:50]}')
                    print(f'    {rec["text"][:150]}')
                    print()

## 4. Clean all manual data

- Hindi manual: ratio 0.40, min 40 chars
- Nepali manual (OCR): ratio 0.20, min 15 chars (OCR is noisy)
- Nepali manual (scraped): ratio 0.30, min 30 chars

In [ ]:
# Clean HINDI MANUAL
import glob, subprocess, sys, os, shutil
out = '/content/drive/MyDrive/LMA_Project/hindi/processed/clean_manual'
if os.path.isdir(out): shutil.rmtree(out)
for f in sorted(glob.glob('/content/drive/MyDrive/LMA_Project/hindi/raw/manual/**/*.jsonl', recursive=True)):
    print('cleaning', os.path.basename(f))
    subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'hindi',
                    '--input', f, '--output-dir', out,
                    '--source-tag', 'manual', '--min-chars', '40',
                    '--min-devanagari-ratio', '0.40'], cwd='/content/lma_code')
print('Hindi manual done')

In [ ]:
# Clean NEPALI MANUAL
import glob, subprocess, sys, os, shutil
out = '/content/drive/MyDrive/LMA_Project/nepali/processed/clean_manual'
if os.path.isdir(out): shutil.rmtree(out)
manual_dirs = [
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/sahityaras/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/cdc/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/national_library/ocr',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/national_library/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/wikisource',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual',  # for news_*.jsonl
]
for d in manual_dirs:
    if not os.path.isdir(d): continue
    for f in glob.glob(os.path.join(d, '*.jsonl')):
        print('cleaning', os.path.basename(f))
        # Use looser filters for OCR dirs, normal for scraped
        is_ocr = 'ocr' in d or 'national_library' in d
        ratio = '0.20' if is_ocr else '0.30'
        minc = '15' if is_ocr else '30'
        subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'nepali',
                        '--input', f, '--output-dir', out,
                        '--source-tag', 'manual', '--min-chars', minc,
                        '--min-devanagari-ratio', ratio], cwd='/content/lma_code')
print('Nepali manual done')

In [ ]:
# Check all cleaned sizes
import os, glob
PROJ = '/content/drive/MyDrive/LMA_Project'
for lang in ['hindi', 'nepali']:
    print(f'\n===== {lang.upper()} =====')
    for subdir in ['processed/clean', 'processed/clean_manual']:
        d = f'{PROJ}/{lang}/{subdir}'
        if not os.path.isdir(d): continue
        total = 0; files = 0
        for f in glob.glob(os.path.join(d, '*.jsonl')):
            total += os.path.getsize(f); files += 1
        print(f'  {subdir}: {files} files, {total/1e6:.1f} MB')

## 5. Balance: cap downloaded to match manual (20% rule)

manual = 20% of total -> downloaded_capped = manual x 4

In [ ]:
# Measure and plan
import os, glob, json
PROJ = '/content/drive/MyDrive/LMA_Project'
for lang in ['hindi', 'nepali']:
    print(f'\n===== {lang.upper()} =====')
    for label, subdir in [('downloaded', 'processed/clean'), ('manual', 'processed/clean_manual')]:
        d = f'{PROJ}/{lang}/{subdir}'
        if not os.path.isdir(d): continue
        total_chars = 0; n_docs = 0
        for f in glob.glob(os.path.join(d, '*.jsonl')):
            with open(f) as fh:
                for line in fh:
                    try:
                        rec = json.loads(line)
                        total_chars += rec.get('n_chars', len(rec.get('text', '')))
                        n_docs += 1
                    except: pass
        print(f'  {label}: {n_docs:,} docs, {total_chars/1e6:.1f}M chars, ~{total_chars/3.5/1e6:.0f}M tokens')

In [ ]:
# Cap downloaded shards
import os, glob, json, shutil
PROJ = '/content/drive/MyDrive/LMA_Project'
for lang in ['hindi', 'nepali']:
    manual_dir = f'{PROJ}/{lang}/processed/clean_manual'
    dl_dir = f'{PROJ}/{lang}/processed/clean'
    capped_dir = f'{PROJ}/{lang}/processed/clean_capped'
    if not os.path.isdir(manual_dir) or not os.path.isdir(dl_dir):
        print(f'{lang}: missing dirs, skipping'); continue
    manual_chars = 0
    for f in glob.glob(f'{manual_dir}/*.jsonl'):
        with open(f) as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                    manual_chars += rec.get('n_chars', len(rec.get('text', '')))
                except: pass
    target_dl = manual_chars * 4
    print(f'\n{lang}: manual={manual_chars/1e6:.1f}M chars, target downloaded={target_dl/1e6:.1f}M chars')
    os.makedirs(capped_dir, exist_ok=True)
    for f in glob.glob(f'{capped_dir}/*.jsonl'):
        os.remove(f)
    accumulated = 0
    for f in sorted(glob.glob(f'{dl_dir}/*.jsonl')):
        if accumulated >= target_dl: break
        dst = os.path.join(capped_dir, os.path.basename(f))
        shutil.copy2(f, dst)
        with open(f) as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                    accumulated += rec.get('n_chars', len(rec.get('text', '')))
                except: pass
        print(f'  copied {os.path.basename(f)} ({accumulated/1e6:.1f}M accumulated)')
    total = manual_chars + accumulated
    if total > 0:
        print(f'{lang}: TOTAL ~{total/1e6:.1f}M chars, manual={manual_chars/total*100:.1f}%, ~{total/3.5/1e6:.0f}M tokens')
    else:
        print(f'{lang}: no data!')

## 6. Merge + splits

In [ ]:
import os, glob
for lang in ['hindi', 'nepali']:
    merged = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/merged'
    os.makedirs(merged, exist_ok=True)
    for f in glob.glob(os.path.join(merged, '*.jsonl')):
        os.remove(f)
    for src_dir in [f'/content/drive/MyDrive/LMA_Project/{lang}/processed/clean_capped',
                    f'/content/drive/MyDrive/LMA_Project/{lang}/processed/clean_manual']:
        if os.path.isdir(src_dir):
            for f in glob.glob(os.path.join(src_dir, '*.jsonl')):
                dst = os.path.join(merged, os.path.basename(f))
                if not os.path.exists(dst):
                    os.symlink(os.path.abspath(f), dst)
    print(lang, 'merged:', len(os.listdir(merged)), 'files')

In [ ]:
for lang in ['hindi', 'nepali']:
    inp = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/merged'
    out = f'/content/drive/MyDrive/LMA_Project/{lang}/splits'
    !cd /content/lma_code && python -m common.splits --lang {lang} --input {inp} \
        --output-dir {out} --train 0.98 --val 0.01 --test 0.01 --seed 17

## 7. Train BPE tokenizers (32k vocab)

In [ ]:
!cd /content/lma_code && python -m hindi.tokenizer.train --language hindi \
    --data /content/drive/MyDrive/LMA_Project/hindi/splits/hindi_train.jsonl \
    --vocab-size 32000 --char-budget 80000000 \
    --output /content/drive/MyDrive/LMA_Project/hindi/tokenizer/hindi_tokenizer.json

In [ ]:
!cd /content/lma_code && python -m nepali.tokenizer.train --language nepali \
    --data /content/drive/MyDrive/LMA_Project/nepali/splits/nepali_train.jsonl \
    --vocab-size 32000 --char-budget 80000000 \
    --output /content/drive/MyDrive/LMA_Project/nepali/tokenizer/nepali_tokenizer.json

## 8. Encode + measure tokens + stats

In [ ]:
for lang in ['hindi', 'nepali']:
    splits = f'/content/drive/MyDrive/LMA_Project/{lang}/splits'
    tok = f'/content/drive/MyDrive/LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json'
    heldout = f'{splits}/{lang}_heldout.jsonl'
    out = f'/content/drive/MyDrive/LMA_Project/{lang}/stats/{lang}_stats.json'
    !cd /content/lma_code && python -m common.stats --lang {lang} --input {splits} \
        --output {out} --tokenizer {tok} --heldout {heldout}

In [ ]:
import json, os
for lang in ['hindi', 'nepali']:
    sp = f'/content/drive/MyDrive/LMA_Project/{lang}/stats/{lang}_stats.json'
    if os.path.exists(sp):
        print(f'\n===== {lang.upper()} STATS =====')
        s = json.load(open(sp))
        for k, v in s.items():
            if k != 'source_breakdown':
                print(f'  {k}: {v}')
        if 'source_breakdown' in s:
            print('  source_breakdown:')
            for kk, vv in s['source_breakdown'].items():
                print(f'    {kk}: {vv}')

## 9. Tokenization examples

In [ ]:
import sys, os
sys.path.insert(0, '/content/lma_code')
from nepali.tokenizer.tokenizer import Tokenizer as NeTok
from hindi.tokenizer.tokenizer import Tokenizer as HiTok
examples = {
  'hindi': ['नमस्ते दुनिया, यह एक परीक्षण वाक्य है।', 'भारत एक विशाल और विविध देश है।'],
  'nepali': ['नमस्ते संसार, यो एक परीक्षण वाक्य हो।', 'नेपाल एक सुन्दर हिमालयको देश हो।'],
}
for lang, toks in [('hindi', HiTok), ('nepali', NeTok)]:
    tpath = f'/content/drive/MyDrive/LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json'
    if not os.path.exists(tpath): print(f'{lang}: tokenizer not found'); continue
    tok = toks.load(tpath)
    print(f'\n===== {lang.upper()} tokenization examples =====')
    for ex in examples[lang]:
        ids = tok.encode(ex)
        print(f'text: {ex}')
        print(f'ids : {ids}')
        print(f'dec : {tok.decode(ids)}')
        print(f'round_trip: {tok.verify_round_trip(ex)}')
        print(f'stats: {tok.tokenization_stats(ex)}')
        print()

## 10. Done!
Paste the stats output so I can fill in the Phase 1 report.
Copy stats + tokenizer JSONs to repo on your Mac for git.